In [1]:
"""
Gene-to-m/z Spatial Pattern Matching Pipeline V5
================================================

MAJOR IMPROVEMENTS:
1. Hybrid architecture: CNN + Transformer + GAT
2. All samples processed (not just 2 per group)
3. Multi-scale importance visualization (10%, 20%, 30%, 50%)
4. Cross-modal attention for gene-m/z alignment
5. Proper saving of all visualizations and embeddings
6. Better matching metrics

Architecture:
- CNN: Captures local spatial texture patterns
- Transformer: Captures global context and long-range dependencies  
- GAT: Captures graph-based neighbor relationships
- Fusion: Combines all three for robust spatial signatures

Author: V5 with hybrid architecture
"""

import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torch_geometric.utils import softmax, add_self_loops
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from scipy.stats import pearsonr, spearmanr
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
import pandas as pd
import os
import warnings
from collections import defaultdict
from dataclasses import dataclass
import pickle

warnings.filterwarnings('ignore')


# =============================================================================
# DATA CONFIGURATION
# =============================================================================

MSI_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_2/h5ad_data_processed_4lockmasses_filtered_halfbrain_common/"
MSI_SAMPLE_FILES = [
    "halfbrain_yc_1_filtered_common.h5ad", "halfbrain_yc_2_filtered_common.h5ad",
    "halfbrain_yc_3_filtered_common.h5ad", "halfbrain_yc_4_filtered_common.h5ad",
    "halfbrain_yad_1_filtered_common.h5ad", "halfbrain_yad_2_filtered_common.h5ad",
    "halfbrain_yad_3_filtered_common.h5ad", "halfbrain_yad_4_filtered_common.h5ad",
    "halfbrain_ac_1_filtered_common.h5ad", "halfbrain_ac_2_filtered_common.h5ad",
    "halfbrain_ac_3_filtered_common.h5ad", "halfbrain_ac_4_filtered_common.h5ad",
    "halfbrain_aad_1_filtered_common.h5ad", "halfbrain_aad_2_filtered_common.h5ad",
    "halfbrain_aad_3_filtered_common.h5ad", "halfbrain_aad_4_filtered_common.h5ad"
]
MSI_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

RNA_INPUT_FOLDER = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_top_800/"
RNA_SAMPLE_FILES = [
    "YC_1.h5ad", "YC_2.h5ad", "YC_3.h5ad", "YC_4.h5ad",
    "YAD_1.h5ad", "YAD_2.h5ad", "YAD_3.h5ad", "YAD_4.h5ad",
    "AC_1.h5ad", "AC_2.h5ad", "AC_3.h5ad", "AC_4.h5ad",
    "AAD_1.h5ad", "AAD_2.h5ad", "AAD_3.h5ad", "AAD_4.h5ad"
]
RNA_SAMPLE_IDS = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

AAD_TARGET_GENES = [
    'Thy1', 'App', 'Apoe', 'Mapt'
]

In [2]:

# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class SpatialSignature:
    """Spatial signature with multi-scale features"""
    sample_id: str
    feature_name: str
    feature_type: str
    global_embedding: np.ndarray
    node_embeddings: np.ndarray
    node_importance: np.ndarray
    coordinates: np.ndarray
    raw_values: np.ndarray
    cnn_features: np.ndarray  # Local texture features
    transformer_features: np.ndarray  # Global context features
    gat_features: np.ndarray  # Graph features

In [3]:

# =============================================================================
# SPATIAL FEATURE EXTRACTION
# =============================================================================

def rotate_coordinates(
    coords: np.ndarray,
    angle_degrees: float = 180.0,
    center: Optional[np.ndarray] = None
) -> np.ndarray:
    """
    Rotate coordinates by given angle around center.
    
    Args:
        coords: [N, 2] array of (x, y) coordinates
        angle_degrees: Rotation angle in degrees (default 180 for flip)
        center: Center of rotation. If None, uses centroid.
    
    Returns:
        Rotated coordinates [N, 2]
    """
    if center is None:
        center = coords.mean(axis=0)
    
    # Convert to radians
    angle_rad = np.radians(angle_degrees)
    
    # Rotation matrix
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    rotation_matrix = np.array([
        [cos_a, -sin_a],
        [sin_a, cos_a]
    ])
    
    # Center, rotate, uncenter
    centered = coords - center
    rotated = centered @ rotation_matrix.T
    result = rotated + center
    
    return result


def flip_coordinates(
    coords: np.ndarray,
    flip_x: bool = True,
    flip_y: bool = True
) -> np.ndarray:
    """
    Flip coordinates (equivalent to 180° rotation).
    
    Args:
        coords: [N, 2] array of (x, y) coordinates
        flip_x: Flip along x-axis
        flip_y: Flip along y-axis
    
    Returns:
        Flipped coordinates [N, 2]
    """
    result = coords.copy()
    center = coords.mean(axis=0)
    
    if flip_x:
        result[:, 0] = 2 * center[0] - result[:, 0]
    if flip_y:
        result[:, 1] = 2 * center[1] - result[:, 1]
    
    return result


def align_coordinates_to_reference(
    source_coords: np.ndarray,
    target_coords: np.ndarray,
    source_values: np.ndarray,
    target_values: np.ndarray,
    test_rotations: List[float] = [0, 90, 180, 270]
) -> Tuple[np.ndarray, float, float]:
    """
    Find best rotation to align source to target based on value correlation.
    
    Args:
        source_coords: Source coordinates to rotate
        target_coords: Target reference coordinates
        source_values: Values at source coordinates
        target_values: Values at target coordinates
        test_rotations: Angles to test
    
    Returns:
        best_coords: Aligned source coordinates
        best_angle: Best rotation angle
        best_corr: Correlation at best angle
    """
    from scipy.interpolate import griddata
    from scipy.stats import pearsonr
    
    # Create common grid
    all_coords = np.vstack([source_coords, target_coords])
    x_min, y_min = all_coords.min(axis=0)
    x_max, y_max = all_coords.max(axis=0)
    
    grid_res = 50
    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, grid_res),
        np.linspace(y_min, y_max, grid_res)
    )
    
    # Interpolate target to grid
    target_grid = griddata(target_coords, target_values, (grid_x, grid_y), 
                           method='linear', fill_value=np.nan)
    
    best_corr = -np.inf
    best_angle = 0
    best_coords = source_coords.copy()
    
    for angle in test_rotations:
        # Rotate source
        rotated = rotate_coordinates(source_coords, angle)
        
        # Scale to match target range
        rotated_min = rotated.min(axis=0)
        rotated_max = rotated.max(axis=0)
        target_min = target_coords.min(axis=0)
        target_max = target_coords.max(axis=0)
        
        # Normalize and rescale
        scale = (target_max - target_min) / (rotated_max - rotated_min + 1e-8)
        aligned = (rotated - rotated_min) * scale + target_min
        
        # Interpolate to grid
        source_grid = griddata(aligned, source_values, (grid_x, grid_y),
                               method='linear', fill_value=np.nan)
        
        # Compute correlation
        mask = ~(np.isnan(source_grid) | np.isnan(target_grid))
        if mask.sum() > 10:
            corr, _ = pearsonr(source_grid[mask], target_grid[mask])
            if not np.isnan(corr) and corr > best_corr:
                best_corr = corr
                best_angle = angle
                best_coords = aligned
    
    return best_coords, best_angle, best_corr


def compute_local_features(
    coords: np.ndarray,
    values: np.ndarray,
    n_neighbors: int = 8
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute local spatial features:
    - Local variance (heterogeneity)
    - Gradient magnitude (boundaries)
    - Local mean (smoothed value)
    """
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    n = len(coords)
    local_var = np.zeros(n)
    gradient = np.zeros(n)
    local_mean = np.zeros(n)
    
    for i in range(n):
        neighbor_vals = values[indices[i, 1:]]
        neighbor_dists = distances[i, 1:]
        
        local_mean[i] = np.mean(neighbor_vals)
        local_var[i] = np.var(neighbor_vals)
        
        val_diff = np.abs(neighbor_vals - values[i])
        gradient[i] = np.mean(val_diff / (neighbor_dists + 1e-8))
    
    # Normalize each to [0, 1]
    for arr in [local_var, gradient, local_mean]:
        if arr.max() > arr.min():
            arr[:] = (arr - arr.min()) / (arr.max() - arr.min())
    
    return local_var, gradient, local_mean


def compute_importance(
    coords: np.ndarray,
    values: np.ndarray,
    n_neighbors: int = 8
) -> np.ndarray:
    """Compute biological importance score"""
    local_var, gradient, local_mean = compute_local_features(coords, values, n_neighbors)
    
    # Normalize values
    val_norm = (values - values.min()) / (values.max() - values.min() + 1e-8)
    
    # Importance = combination of variance, gradient, and value
    importance = 0.35 * local_var + 0.30 * gradient + 0.35 * val_norm
    
    return importance

In [4]:

# =============================================================================
# SPATIAL GRAPH CONSTRUCTION
# =============================================================================

def build_spatial_graph(
    coords: np.ndarray,
    n_neighbors: int = 8
) -> torch.Tensor:
    """Build k-NN spatial graph"""
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1)
    nn.fit(coords)
    distances, indices = nn.kneighbors(coords)
    
    # Adaptive threshold
    median_dist = np.median(distances[:, 1])
    max_dist = median_dist * 1.5
    
    edge_list = []
    for i in range(len(coords)):
        for j_idx in range(1, n_neighbors + 1):
            j = indices[i, j_idx]
            if distances[i, j_idx] <= max_dist:
                edge_list.append([i, j])
                edge_list.append([j, i])
    
    # Remove duplicates
    edge_set = set(map(tuple, edge_list))
    edge_list = list(edge_set)
    
    if len(edge_list) == 0:
        # Fallback
        for i in range(len(coords)):
            for j in indices[i, 1:n_neighbors+1]:
                edge_list.append([i, j])
                edge_list.append([j, i])
    
    return torch.tensor(edge_list, dtype=torch.long).t().contiguous()

In [5]:

# =============================================================================
# CNN MODULE FOR LOCAL TEXTURE
# =============================================================================

class SpatialCNN(nn.Module):
    """
    1D CNN applied to local neighborhood features.
    Captures local texture patterns around each node.
    """
    
    def __init__(self, input_dim: int, hidden_dim: int = 64, output_dim: int = 32):
        super().__init__()
        
        self.conv1 = nn.Conv1d(1, hidden_dim, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(hidden_dim, output_dim, kernel_size=3, padding=1)
        
        self.norm1 = nn.BatchNorm1d(hidden_dim)
        self.norm2 = nn.BatchNorm1d(hidden_dim)
        self.norm3 = nn.BatchNorm1d(output_dim)
        
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        # Project to output dim
        self.project = nn.Linear(output_dim, output_dim)
    
    def forward(self, x: torch.Tensor, neighbor_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Node features [N, input_dim]
            neighbor_features: Aggregated neighbor features [N, input_dim]
        
        Returns:
            CNN features [N, output_dim]
        """
        # Concatenate self and neighbor features, treat as 1D signal
        # Shape: [N, 1, 2*input_dim]
        combined = torch.cat([x, neighbor_features], dim=-1).unsqueeze(1)
        
        h = F.gelu(self.norm1(self.conv1(combined)))
        h = F.gelu(self.norm2(self.conv2(h)))
        h = F.gelu(self.norm3(self.conv3(h)))
        
        h = self.pool(h).squeeze(-1)  # [N, output_dim]
        h = self.project(h)
        
        return h

In [6]:

# =============================================================================
# TRANSFORMER MODULE FOR GLOBAL CONTEXT
# =============================================================================

class SpatialTransformer(nn.Module):
    """
    Transformer for capturing global spatial context.
    Uses positional encoding based on spatial coordinates.
    """
    
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 128,
        output_dim: int = 64,
        num_heads: int = 4,
        num_layers: int = 2,
        max_nodes: int = 10000
    ):
        super().__init__()
        
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        
        # Learnable spatial positional encoding
        self.pos_encoder = nn.Sequential(
            nn.Linear(2, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, hidden_dim)
        )
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.output_projection = nn.Linear(hidden_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
    
    def forward(
        self,
        x: torch.Tensor,
        pos: torch.Tensor,
        batch_size: int = 2000
    ) -> torch.Tensor:
        """
        Args:
            x: Node features [N, input_dim]
            pos: Spatial positions [N, 2]
            batch_size: Process in batches for memory efficiency
        
        Returns:
            Transformer features [N, output_dim]
        """
        N = x.size(0)
        
        # Project input
        h = self.input_projection(x)  # [N, hidden_dim]
        
        # Add positional encoding
        pos_enc = self.pos_encoder(pos)  # [N, hidden_dim]
        h = h + pos_enc
        
        # Process in batches if too large
        if N > batch_size:
            # Sample subset for global context
            indices = torch.randperm(N, device=x.device)[:batch_size]
            h_subset = h[indices].unsqueeze(0)  # [1, batch_size, hidden_dim]
            
            h_transformed = self.transformer(h_subset).squeeze(0)  # [batch_size, hidden_dim]
            
            # For nodes not in subset, use nearest neighbor from subset
            out = torch.zeros(N, h_transformed.size(-1), device=x.device)
            out[indices] = h_transformed
            
            # Interpolate for other nodes
            mask = torch.ones(N, dtype=torch.bool, device=x.device)
            mask[indices] = False
            
            if mask.sum() > 0:
                # Use k-NN to interpolate
                pos_subset = pos[indices]
                pos_other = pos[mask]
                
                # Compute distances
                dist = torch.cdist(pos_other, pos_subset)
                k = min(5, batch_size)
                _, nearest = dist.topk(k, dim=1, largest=False)
                
                # Average features from nearest neighbors
                out[mask] = h_transformed[nearest].mean(dim=1)
        else:
            h = h.unsqueeze(0)  # [1, N, hidden_dim]
            h_transformed = self.transformer(h).squeeze(0)  # [N, hidden_dim]
            out = h_transformed
        
        out = self.output_projection(out)
        out = self.norm(out)
        
        return out

In [7]:

# =============================================================================
# GAT MODULE FOR GRAPH STRUCTURE
# =============================================================================

class SpatialGAT(nn.Module):
    """
    Graph Attention Network for capturing local graph structure.
    """
    
    def __init__(
        self,
        input_dim: int,
        hidden_dim: int = 64,
        output_dim: int = 32,
        num_heads: int = 4,
        num_layers: int = 2,
        dropout: float = 0.1
    ):
        super().__init__()
        
        self.input_projection = nn.Linear(input_dim, hidden_dim)
        
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        
        # First layer
        self.convs.append(GATConv(
            hidden_dim, hidden_dim // num_heads, 
            heads=num_heads, concat=True, dropout=dropout
        ))
        self.norms.append(nn.LayerNorm(hidden_dim))
        
        # Additional layers
        for _ in range(num_layers - 1):
            self.convs.append(GATConv(
                hidden_dim, hidden_dim // num_heads,
                heads=num_heads, concat=True, dropout=dropout
            ))
            self.norms.append(nn.LayerNorm(hidden_dim))
        
        self.output_projection = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor
    ) -> Tuple[torch.Tensor, List[torch.Tensor]]:
        """
        Args:
            x: Node features [N, input_dim]
            edge_index: Graph edges [2, E]
        
        Returns:
            GAT features [N, output_dim]
            Attention weights list
        """
        h = self.input_projection(x)
        
        attention_weights = []
        
        for conv, norm in zip(self.convs, self.norms):
            h_new, attn = conv(h, edge_index, return_attention_weights=True)
            attention_weights.append(attn[1])
            h_new = norm(h_new)
            h_new = F.gelu(h_new)
            h_new = self.dropout(h_new)
            h = h_new
        
        out = self.output_projection(h)
        
        return out, attention_weights

In [8]:

# =============================================================================
# HYBRID SPATIAL ENCODER
# =============================================================================

class HybridSpatialEncoder(nn.Module):
    """
    Hybrid encoder combining CNN, Transformer, and GAT.
    
    - CNN: Local texture patterns
    - Transformer: Global context
    - GAT: Graph-based neighbor relationships
    """
    
    def __init__(
        self,
        input_dim: int = None,
        hidden_dim: int = 128,
        embedding_dim: int = 64,
        projection_dim: int = 64
    ):
        super().__init__()
        
        self.projection_dim = projection_dim
        self.embedding_dim = embedding_dim
        
        # Input projections for different dimensions
        self.input_projections = nn.ModuleDict()
        if input_dim is not None:
            self.input_projections[str(input_dim)] = nn.Sequential(
                nn.Linear(input_dim, projection_dim),
                nn.LayerNorm(projection_dim),
                nn.GELU()
            )
        
        # Sub-modules
        self.cnn = SpatialCNN(projection_dim, hidden_dim // 2, embedding_dim // 3)
        self.transformer = SpatialTransformer(projection_dim, hidden_dim, embedding_dim // 3)
        self.gat = SpatialGAT(projection_dim, hidden_dim // 2, embedding_dim // 3)
        
        # Fusion layer
        combined_dim = 3 * (embedding_dim // 3)
        self.fusion = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, embedding_dim)
        )
        
        # Importance head
        self.importance_head = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.Sigmoid()
        )
        
        # Global pooling
        self.pool_attention = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Reconstruction decoder
        self.decoder = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, projection_dim)
        )
    
    def _get_projection(self, input_dim: int) -> nn.Module:
        key = str(input_dim)
        if key not in self.input_projections:
            device = next(self.parameters()).device
            self.input_projections[key] = nn.Sequential(
                nn.Linear(input_dim, self.projection_dim),
                nn.LayerNorm(self.projection_dim),
                nn.GELU()
            ).to(device)
        return self.input_projections[key]
    
    def _aggregate_neighbors(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor
    ) -> torch.Tensor:
        """Aggregate neighbor features for CNN input"""
        N = x.size(0)
        src, dst = edge_index[0], edge_index[1]
        
        # Sum aggregation
        neighbor_sum = torch.zeros(N, x.size(1), device=x.device)
        neighbor_count = torch.zeros(N, 1, device=x.device)
        
        neighbor_sum.scatter_add_(0, dst.view(-1, 1).expand(-1, x.size(1)), x[src])
        neighbor_count.scatter_add_(0, dst.view(-1, 1), torch.ones(len(dst), 1, device=x.device))
        
        neighbor_mean = neighbor_sum / (neighbor_count + 1e-8)
        
        return neighbor_mean
    
    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        pos: torch.Tensor,
        bio_importance: Optional[torch.Tensor] = None
    ) -> dict:
        """Forward pass through hybrid encoder"""
        
        # Project input
        h_input = self._get_projection(x.size(1))(x)
        
        # Aggregate neighbors for CNN
        neighbor_features = self._aggregate_neighbors(h_input, edge_index)
        
        # CNN branch: local texture
        cnn_out = self.cnn(h_input, neighbor_features)
        
        # Transformer branch: global context
        transformer_out = self.transformer(h_input, pos)
        
        # GAT branch: graph structure
        gat_out, attention_weights = self.gat(h_input, edge_index)
        
        # Fusion
        combined = torch.cat([cnn_out, transformer_out, gat_out], dim=-1)
        node_embeddings = self.fusion(combined)
        
        # Learned importance
        learned_importance = self.importance_head(node_embeddings).squeeze(-1)
        
        # Combine with biological importance
        if bio_importance is not None:
            node_importance = 0.5 * learned_importance + 0.5 * bio_importance
        else:
            node_importance = learned_importance
        
        node_importance = node_importance / (node_importance.max() + 1e-8)
        
        # Weighted embeddings
        weight = node_importance / (node_importance.sum() + 1e-8) * x.size(0)
        weighted_embeddings = node_embeddings * weight.unsqueeze(-1)
        
        # Global embedding
        attn_scores = self.pool_attention(node_embeddings).squeeze(-1)
        attn_weights = F.softmax(attn_scores, dim=0)
        global_embedding = (attn_weights.unsqueeze(-1) * weighted_embeddings).sum(dim=0)
        
        # Reconstruction
        reconstructed = self.decoder(node_embeddings)
        
        return {
            'node_embeddings': node_embeddings,
            'weighted_embeddings': weighted_embeddings,
            'global_embedding': global_embedding,
            'node_importance': node_importance,
            'cnn_features': cnn_out,
            'transformer_features': transformer_out,
            'gat_features': gat_out,
            'attention_weights': attention_weights,
            'reconstructed': reconstructed,
            'h_input': h_input
        }

In [9]:

# =============================================================================
# LOSS FUNCTION
# =============================================================================

class HybridLoss(nn.Module):
    """Loss function for hybrid encoder"""
    
    def __init__(self):
        super().__init__()
    
    def forward(
        self,
        output: dict,
        pos: torch.Tensor,
        edge_index: torch.Tensor
    ) -> Tuple[torch.Tensor, dict]:
        
        losses = {}
        
        # 1. Reconstruction
        recon_loss = F.mse_loss(output['reconstructed'], output['h_input'])
        losses['reconstruction'] = recon_loss
        
        # 2. Spatial smoothness
        src, dst = edge_index[0], edge_index[1]
        emb_diff = output['node_embeddings'][src] - output['node_embeddings'][dst]
        smooth_loss = (emb_diff ** 2).mean() / output['node_embeddings'].size(-1)
        losses['smoothness'] = smooth_loss * 0.3
        
        # 3. Structure preservation
        N = pos.size(0)
        n_samples = min(1000, N * (N - 1) // 2)
        
        idx1 = torch.randint(0, N, (n_samples,), device=pos.device)
        idx2 = torch.randint(0, N, (n_samples,), device=pos.device)
        mask = idx1 != idx2
        idx1, idx2 = idx1[mask], idx2[mask]
        
        if len(idx1) > 10:
            spatial_dist = torch.norm(pos[idx1] - pos[idx2], dim=-1)
            emb_dist = torch.norm(
                output['node_embeddings'][idx1] - output['node_embeddings'][idx2],
                dim=-1
            )
            
            spatial_dist = spatial_dist / (spatial_dist.max() + 1e-8)
            emb_dist = emb_dist / (emb_dist.max() + 1e-8)
            
            structure_loss = F.mse_loss(emb_dist, spatial_dist)
            losses['structure'] = structure_loss * 0.3
        else:
            losses['structure'] = torch.tensor(0.0, device=pos.device)
        
        # 4. Feature consistency across branches
        cnn_norm = F.normalize(output['cnn_features'], dim=-1)
        trans_norm = F.normalize(output['transformer_features'], dim=-1)
        gat_norm = F.normalize(output['gat_features'], dim=-1)
        
        consistency_loss = (
            (1 - (cnn_norm * trans_norm).sum(dim=-1)).mean() +
            (1 - (cnn_norm * gat_norm).sum(dim=-1)).mean() +
            (1 - (trans_norm * gat_norm).sum(dim=-1)).mean()
        ) / 3
        losses['consistency'] = consistency_loss * 0.1
        
        total = sum(losses.values())
        loss_dict = {k: v.item() for k, v in losses.items()}
        loss_dict['total'] = total.item()
        
        return total, loss_dict

In [10]:

# =============================================================================
# MAIN MATCHER
# =============================================================================

class HybridMatcher:
    """Gene-to-m/z matcher with hybrid architecture"""
    
    def __init__(
        self,
        output_dir: str = './gene_to_mz_results_v5',
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        self.device = device
        self.output_dir = output_dir
        
        # Create ALL output directories
        subdirs = [
            'saliency_maps', 'gene_visualizations', 'training_curves',
            'embeddings', 'match_visualizations', 'importance_maps'
        ]
        for subdir in subdirs:
            os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)
        
        self.rna_data = {}
        self.msi_data = {}
        self.rna_model = None
        self.msi_model = None
        
        self.gene_signatures = defaultdict(dict)
        self.mz_signatures = defaultdict(dict)
    
    def load_all_data(self):
        """Load all data"""
        print("Loading RNA-seq data...")
        for file, sample_id in zip(RNA_SAMPLE_FILES, RNA_SAMPLE_IDS):
            path = os.path.join(RNA_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.rna_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.rna_data[sample_id].shape}")
        
        print("\nLoading MSI data...")
        for file, sample_id in zip(MSI_SAMPLE_FILES, MSI_SAMPLE_IDS):
            path = os.path.join(MSI_INPUT_FOLDER, file)
            if os.path.exists(path):
                self.msi_data[sample_id] = sc.read_h5ad(path)
                print(f"  {sample_id}: {self.msi_data[sample_id].shape}")
    
    def check_gene_availability(self) -> Dict[str, Dict[str, bool]]:
        """Check gene availability"""
        gene_availability = {}
        for gene in AAD_TARGET_GENES:
            gene_availability[gene] = {}
            for sample_id in RNA_SAMPLE_IDS:
                if sample_id in self.rna_data:
                    gene_availability[gene][sample_id] = gene in self.rna_data[sample_id].var_names
                else:
                    gene_availability[gene][sample_id] = False
        
        print("\nGene availability:")
        for gene in AAD_TARGET_GENES:
            available = sum(gene_availability[gene].values())
            print(f"  {gene}: {available}/{len(RNA_SAMPLE_IDS)} samples")
        
        return gene_availability
    
    def prepare_data(
        self,
        coords: np.ndarray,
        features: np.ndarray,
        n_neighbors: int
    ) -> Tuple[Data, np.ndarray]:
        """Prepare graph data"""
        if features.ndim == 1:
            features = features.reshape(-1, 1)
        
        # Compute biological importance
        bio_importance = np.zeros(len(coords))
        for col in range(features.shape[1]):
            bio_importance += compute_importance(coords, features[:, col], n_neighbors)
        bio_importance /= features.shape[1]
        
        # Scale features
        scaler = RobustScaler()
        features_scaled = scaler.fit_transform(features)
        
        # Build graph
        edge_index = build_spatial_graph(coords, n_neighbors)
        
        # Normalize positions
        pos = (coords - coords.mean(axis=0)) / (coords.std(axis=0) + 1e-8)
        
        data = Data(
            x=torch.tensor(features_scaled, dtype=torch.float32),
            edge_index=edge_index,
            pos=torch.tensor(pos, dtype=torch.float32)
        )
        
        return data, bio_importance
    
    def train_model(
        self,
        data_dict: Dict[str, Data],
        bio_importance_dict: Dict[str, np.ndarray],
        model_type: str,
        epochs: int = 150
    ) -> HybridSpatialEncoder:
        """Train hybrid model"""
        print(f"\nTraining {model_type.upper()} model...")
        
        first_data = list(data_dict.values())[0]
        input_dim = first_data.x.size(1)
        
        model = HybridSpatialEncoder(
            input_dim=input_dim,
            hidden_dim=128,
            embedding_dim=64,
            projection_dim=64
        ).to(self.device)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        warmup = 10
        def lr_lambda(epoch):
            if epoch < warmup:
                return (epoch + 1) / warmup
            progress = (epoch - warmup) / (epochs - warmup)
            return 0.5 * (1 + np.cos(np.pi * progress))
        
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        loss_fn = HybridLoss()
        
        model.train()
        loss_history = []
        
        for epoch in range(epochs):
            total_loss = 0
            
            for sample_id, data in data_dict.items():
                data = data.to(self.device)
                bio_imp = torch.tensor(
                    bio_importance_dict[sample_id],
                    dtype=torch.float32,
                    device=self.device
                )
                
                optimizer.zero_grad()
                output = model(data.x, data.edge_index, data.pos, bio_imp)
                loss, _ = loss_fn(output, data.pos, data.edge_index)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                
                total_loss += loss.item()
            
            scheduler.step()
            avg_loss = total_loss / len(data_dict)
            loss_history.append(avg_loss)
            
            if (epoch + 1) % 30 == 0:
                print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
        
        print(f"  Final loss: {loss_history[-1]:.4f}")
        
        # Save loss plot
        plt.figure(figsize=(10, 5))
        plt.plot(loss_history)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'{model_type.upper()} Training Loss')
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(self.output_dir, 'training_curves', f'{model_type}_loss.png'), dpi=150)
        plt.close()
        
        return model
    
    def extract_gene_signature(
        self,
        sample_id: str,
        gene: str,
        model: HybridSpatialEncoder
    ) -> Optional[SpatialSignature]:
        """Extract gene signature"""
        if sample_id not in self.rna_data:
            return None
        
        adata = self.rna_data[sample_id]
        if gene not in adata.var_names:
            return None
        
        coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
        
        gene_idx = list(adata.var_names).index(gene)
        if hasattr(adata.X, 'toarray'):
            gene_expr = adata.X[:, gene_idx].toarray().flatten()
        else:
            gene_expr = adata.X[:, gene_idx].flatten()
        
        bio_importance = compute_importance(coords, gene_expr, n_neighbors=6)
        
        data, _ = self.prepare_data(coords, gene_expr, n_neighbors=6)
        data = data.to(self.device)
        bio_imp_tensor = torch.tensor(bio_importance, dtype=torch.float32, device=self.device)
        
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, data.pos, bio_imp_tensor)
        
        sig = SpatialSignature(
            sample_id=sample_id,
            feature_name=gene,
            feature_type='gene',
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=output['weighted_embeddings'].cpu().numpy(),
            node_importance=output['node_importance'].cpu().numpy(),
            coordinates=coords,
            raw_values=gene_expr,
            cnn_features=output['cnn_features'].cpu().numpy(),
            transformer_features=output['transformer_features'].cpu().numpy(),
            gat_features=output['gat_features'].cpu().numpy()
        )
        
        self.gene_signatures[gene][sample_id] = sig
        return sig
    
    def extract_mz_signature(
        self,
        sample_id: str,
        mz_name: str,
        mz_values: np.ndarray,
        coords: np.ndarray,
        model: HybridSpatialEncoder
    ) -> SpatialSignature:
        """Extract m/z signature"""
        bio_importance = compute_importance(coords, mz_values, n_neighbors=8)
        
        data, _ = self.prepare_data(coords, mz_values, n_neighbors=8)
        data = data.to(self.device)
        bio_imp_tensor = torch.tensor(bio_importance, dtype=torch.float32, device=self.device)
        
        model.eval()
        with torch.no_grad():
            output = model(data.x, data.edge_index, data.pos, bio_imp_tensor)
        
        return SpatialSignature(
            sample_id=sample_id,
            feature_name=mz_name,
            feature_type='mz',
            global_embedding=output['global_embedding'].cpu().numpy(),
            node_embeddings=output['weighted_embeddings'].cpu().numpy(),
            node_importance=output['node_importance'].cpu().numpy(),
            coordinates=coords,
            raw_values=mz_values,
            cnn_features=output['cnn_features'].cpu().numpy(),
            transformer_features=output['transformer_features'].cpu().numpy(),
            gat_features=output['gat_features'].cpu().numpy()
        )
    
    def compute_similarity(
        self,
        gene_sig: SpatialSignature,
        mz_sig: SpatialSignature
    ) -> dict:
        """Compute multi-modal similarity"""
        
        # Global embedding similarity
        g_emb = gene_sig.global_embedding.flatten()
        m_emb = mz_sig.global_embedding.flatten()
        emb_cosine = np.dot(g_emb, m_emb) / (np.linalg.norm(g_emb) * np.linalg.norm(m_emb) + 1e-8)
        
        # CNN feature similarity
        g_cnn = gene_sig.cnn_features.mean(axis=0)
        m_cnn = mz_sig.cnn_features.mean(axis=0)
        cnn_cosine = np.dot(g_cnn, m_cnn) / (np.linalg.norm(g_cnn) * np.linalg.norm(m_cnn) + 1e-8)
        
        # Transformer feature similarity
        g_trans = gene_sig.transformer_features.mean(axis=0)
        m_trans = mz_sig.transformer_features.mean(axis=0)
        trans_cosine = np.dot(g_trans, m_trans) / (np.linalg.norm(g_trans) * np.linalg.norm(m_trans) + 1e-8)
        
        # GAT feature similarity
        g_gat = gene_sig.gat_features.mean(axis=0)
        m_gat = mz_sig.gat_features.mean(axis=0)
        gat_cosine = np.dot(g_gat, m_gat) / (np.linalg.norm(g_gat) * np.linalg.norm(m_gat) + 1e-8)
        
        # Importance overlap
        imp_overlap = self._compute_importance_overlap(gene_sig, mz_sig)
        
        # Raw correlation
        raw_pearson = self._compute_raw_correlation(gene_sig, mz_sig)
        
        return {
            'embedding_cosine': emb_cosine,
            'cnn_cosine': cnn_cosine,
            'transformer_cosine': trans_cosine,
            'gat_cosine': gat_cosine,
            'importance_overlap': imp_overlap,
            'raw_pearson': raw_pearson
        }
    
    def _compute_importance_overlap(self, sig1, sig2, grid_res=50, align=True):
        """Compute importance overlap with optional rotation alignment"""
        
        # For gene-to-mz comparison, align gene (RNA) to mz (MSI)
        if align and sig1.feature_type == 'gene' and sig2.feature_type == 'mz':
            # Apply 180° rotation to RNA coordinates
            aligned_coords = rotate_coordinates(sig1.coordinates, angle_degrees=180)
            
            # Scale to match MSI coordinate range
            src_min, src_max = aligned_coords.min(axis=0), aligned_coords.max(axis=0)
            tgt_min, tgt_max = sig2.coordinates.min(axis=0), sig2.coordinates.max(axis=0)
            
            scale = (tgt_max - tgt_min) / (src_max - src_min + 1e-8)
            aligned_coords = (aligned_coords - src_min) * scale + tgt_min
        else:
            aligned_coords = sig1.coordinates
        
        x_min = min(aligned_coords[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(aligned_coords[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(aligned_coords[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(aligned_coords[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        imp1 = griddata(aligned_coords, sig1.node_importance, (grid_x, grid_y), method='linear', fill_value=0)
        imp2 = griddata(sig2.coordinates, sig2.node_importance, (grid_x, grid_y), method='linear', fill_value=0)
        
        imp1 = imp1 / (imp1.max() + 1e-8)
        imp2 = imp2 / (imp2.max() + 1e-8)
        
        intersection = np.minimum(imp1, imp2).sum()
        union = np.maximum(imp1, imp2).sum()
        
        return intersection / (union + 1e-8)
    
    def _compute_raw_correlation(self, sig1, sig2, grid_res=50, align=True):
        """Compute raw correlation with rotation alignment"""
        
        # For gene-to-mz comparison, align gene (RNA) to mz (MSI)
        if align and sig1.feature_type == 'gene' and sig2.feature_type == 'mz':
            # Apply 180° rotation to RNA coordinates
            aligned_coords = rotate_coordinates(sig1.coordinates, angle_degrees=180)
            
            # Scale to match MSI coordinate range
            src_min, src_max = aligned_coords.min(axis=0), aligned_coords.max(axis=0)
            tgt_min, tgt_max = sig2.coordinates.min(axis=0), sig2.coordinates.max(axis=0)
            
            scale = (tgt_max - tgt_min) / (src_max - src_min + 1e-8)
            aligned_coords = (aligned_coords - src_min) * scale + tgt_min
        else:
            aligned_coords = sig1.coordinates
        
        x_min = min(aligned_coords[:, 0].min(), sig2.coordinates[:, 0].min())
        x_max = max(aligned_coords[:, 0].max(), sig2.coordinates[:, 0].max())
        y_min = min(aligned_coords[:, 1].min(), sig2.coordinates[:, 1].min())
        y_max = max(aligned_coords[:, 1].max(), sig2.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(
            np.linspace(x_min, x_max, grid_res),
            np.linspace(y_min, y_max, grid_res)
        )
        
        v1 = griddata(aligned_coords, sig1.raw_values, (grid_x, grid_y), method='linear', fill_value=np.nan)
        v2 = griddata(sig2.coordinates, sig2.raw_values, (grid_x, grid_y), method='linear', fill_value=np.nan)
        
        mask = ~(np.isnan(v1) | np.isnan(v2))
        if mask.sum() < 10:
            return 0.0
        
        r, _ = pearsonr(v1[mask], v2[mask])
        return r if not np.isnan(r) else 0.0
    
    def find_matches(self, gene_sig, mz_signatures, top_k=20):
        """Find best matches with multi-modal scoring"""
        matches = []
        
        for mz_name, mz_sig in mz_signatures.items():
            sim = self.compute_similarity(gene_sig, mz_sig)
            
            # Combined score using all modalities
            combined = (
                0.25 * sim['embedding_cosine'] +
                0.15 * sim['cnn_cosine'] +
                0.15 * sim['transformer_cosine'] +
                0.15 * sim['gat_cosine'] +
                0.15 * sim['importance_overlap'] +
                0.15 * max(sim['raw_pearson'], 0)
            )
            
            matches.append({
                'gene': gene_sig.feature_name,
                'rna_sample': gene_sig.sample_id,
                'mz_feature': mz_name,
                'msi_sample': mz_sig.sample_id,
                **sim,
                'combined_score': combined
            })
        
        df = pd.DataFrame(matches)
        return df.sort_values('combined_score', ascending=False).head(top_k)
    
    def visualize_signature(self, sig: SpatialSignature, save_path: str):
        """Visualize with multi-scale importance"""
        fig, axes = plt.subplots(2, 4, figsize=(20, 10))
        
        # Row 1: Raw and importance
        im1 = axes[0, 0].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.raw_values, cmap='viridis', s=10, alpha=0.8)
        axes[0, 0].set_title(f'Raw: {sig.feature_name}', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        im2 = axes[0, 1].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=sig.node_importance, cmap='hot', s=10, alpha=0.8)
        axes[0, 1].set_title('Importance', fontweight='bold')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # Importance-weighted
        weighted = sig.raw_values * sig.node_importance
        weighted = (weighted - weighted.min()) / (weighted.max() - weighted.min() + 1e-8)
        im3 = axes[0, 2].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=weighted, cmap='magma', s=10, alpha=0.8)
        axes[0, 2].set_title('Importance-Weighted', fontweight='bold')
        plt.colorbar(im3, ax=axes[0, 2])
        
        # Feature norm (CNN + Transformer + GAT)
        feature_norm = (
            np.linalg.norm(sig.cnn_features, axis=1) +
            np.linalg.norm(sig.transformer_features, axis=1) +
            np.linalg.norm(sig.gat_features, axis=1)
        )
        feature_norm = (feature_norm - feature_norm.min()) / (feature_norm.max() - feature_norm.min() + 1e-8)
        im4 = axes[0, 3].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                  c=feature_norm, cmap='plasma', s=10, alpha=0.8)
        axes[0, 3].set_title('Feature Magnitude', fontweight='bold')
        plt.colorbar(im4, ax=axes[0, 3])
        
        # Row 2: Multi-scale important regions (10%, 20%, 30%, 50%)
        thresholds = [90, 80, 70, 50]
        titles = ['Top 10%', 'Top 20%', 'Top 30%', 'Top 50%']
        cmaps = ['Reds', 'Oranges', 'YlOrBr', 'YlGn']
        
        for idx, (thresh, title, cmap) in enumerate(zip(thresholds, titles, cmaps)):
            threshold_val = np.percentile(sig.node_importance, thresh)
            mask = sig.node_importance >= threshold_val
            
            axes[1, idx].scatter(sig.coordinates[:, 0], sig.coordinates[:, 1],
                                 c='lightgray', s=5, alpha=0.3)
            if mask.sum() > 0:
                sc = axes[1, idx].scatter(sig.coordinates[mask, 0], sig.coordinates[mask, 1],
                                          c=sig.raw_values[mask], cmap=cmap, s=15, alpha=0.9)
                plt.colorbar(sc, ax=axes[1, idx])
            axes[1, idx].set_title(f'{title} ({mask.sum()} pts)', fontweight='bold')
        
        plt.suptitle(f'{sig.feature_type.upper()}: {sig.feature_name} ({sig.sample_id})',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def visualize_match(self, gene_sig, mz_sig, sim, save_path):
        """Visualize gene-m/z match with rotation alignment"""
        fig, axes = plt.subplots(3, 4, figsize=(20, 15))
        
        # Row 0: Original gene visualizations
        im1 = axes[0, 0].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                  c=gene_sig.raw_values, cmap='viridis', s=15)
        axes[0, 0].set_title(f'Gene: {gene_sig.feature_name} (Original)', fontweight='bold')
        plt.colorbar(im1, ax=axes[0, 0])
        
        im2 = axes[0, 1].scatter(gene_sig.coordinates[:, 0], gene_sig.coordinates[:, 1],
                                  c=gene_sig.node_importance, cmap='hot', s=15)
        axes[0, 1].set_title('Gene Importance (Original)', fontweight='bold')
        plt.colorbar(im2, ax=axes[0, 1])
        
        # Rotate gene coordinates 180° and scale to MSI range
        aligned_coords = rotate_coordinates(gene_sig.coordinates, angle_degrees=180)
        src_min, src_max = aligned_coords.min(axis=0), aligned_coords.max(axis=0)
        tgt_min, tgt_max = mz_sig.coordinates.min(axis=0), mz_sig.coordinates.max(axis=0)
        scale = (tgt_max - tgt_min) / (src_max - src_min + 1e-8)
        aligned_coords = (aligned_coords - src_min) * scale + tgt_min
        
        # Row 0: Aligned gene visualizations
        im3 = axes[0, 2].scatter(aligned_coords[:, 0], aligned_coords[:, 1],
                                  c=gene_sig.raw_values, cmap='viridis', s=15)
        axes[0, 2].set_title('Gene (Rotated 180°, Aligned)', fontweight='bold')
        plt.colorbar(im3, ax=axes[0, 2])
        
        im4 = axes[0, 3].scatter(aligned_coords[:, 0], aligned_coords[:, 1],
                                  c=gene_sig.node_importance, cmap='hot', s=15)
        axes[0, 3].set_title('Gene Importance (Aligned)', fontweight='bold')
        plt.colorbar(im4, ax=axes[0, 3])
        
        # Row 1: m/z visualizations
        im5 = axes[1, 0].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.raw_values, cmap='viridis', s=5)
        axes[1, 0].set_title(f'm/z: {mz_sig.feature_name}', fontweight='bold')
        plt.colorbar(im5, ax=axes[1, 0])
        
        im6 = axes[1, 1].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                                  c=mz_sig.node_importance, cmap='hot', s=5)
        axes[1, 1].set_title('m/z Importance', fontweight='bold')
        plt.colorbar(im6, ax=axes[1, 1])
        
        # Overlay: Gene (aligned) + m/z
        axes[1, 2].scatter(mz_sig.coordinates[:, 0], mz_sig.coordinates[:, 1],
                          c='blue', s=3, alpha=0.3, label='m/z')
        axes[1, 2].scatter(aligned_coords[:, 0], aligned_coords[:, 1],
                          c='red', s=10, alpha=0.5, label='Gene (aligned)')
        axes[1, 2].set_title('Overlay: Gene (red) + m/z (blue)', fontweight='bold')
        axes[1, 2].legend()
        
        # Importance overlay
        gene_thresh = np.percentile(gene_sig.node_importance, 70)
        mz_thresh = np.percentile(mz_sig.node_importance, 70)
        gene_mask = gene_sig.node_importance >= gene_thresh
        mz_mask = mz_sig.node_importance >= mz_thresh
        
        axes[1, 3].scatter(mz_sig.coordinates[mz_mask, 0], mz_sig.coordinates[mz_mask, 1],
                          c='blue', s=8, alpha=0.5, label='m/z top 30%')
        axes[1, 3].scatter(aligned_coords[gene_mask, 0], aligned_coords[gene_mask, 1],
                          c='red', s=15, alpha=0.7, label='Gene top 30%')
        axes[1, 3].set_title('Top 30% Importance Overlay', fontweight='bold')
        axes[1, 3].legend()
        
        # Row 2: Correlation and metrics
        grid_res = 50
        x_min = min(aligned_coords[:, 0].min(), mz_sig.coordinates[:, 0].min())
        x_max = max(aligned_coords[:, 0].max(), mz_sig.coordinates[:, 0].max())
        y_min = min(aligned_coords[:, 1].min(), mz_sig.coordinates[:, 1].min())
        y_max = max(aligned_coords[:, 1].max(), mz_sig.coordinates[:, 1].max())
        
        grid_x, grid_y = np.meshgrid(np.linspace(x_min, x_max, grid_res),
                                      np.linspace(y_min, y_max, grid_res))
        
        v1 = griddata(aligned_coords, gene_sig.raw_values, (grid_x, grid_y), method='linear')
        v2 = griddata(mz_sig.coordinates, mz_sig.raw_values, (grid_x, grid_y), method='linear')
        
        mask = ~(np.isnan(v1) | np.isnan(v2))
        if mask.sum() > 0:
            axes[2, 0].scatter(v1[mask], v2[mask], alpha=0.3, s=10)
            axes[2, 0].set_xlabel('Gene Expression (aligned)')
            axes[2, 0].set_ylabel('m/z Intensity')
            axes[2, 0].set_title(f'Spatial Correlation (r={sim["raw_pearson"]:.3f})', fontweight='bold')
        
        # Importance correlation
        imp1 = griddata(aligned_coords, gene_sig.node_importance, (grid_x, grid_y), method='linear')
        imp2 = griddata(mz_sig.coordinates, mz_sig.node_importance, (grid_x, grid_y), method='linear')
        
        mask_imp = ~(np.isnan(imp1) | np.isnan(imp2))
        if mask_imp.sum() > 0:
            axes[2, 1].scatter(imp1[mask_imp], imp2[mask_imp], alpha=0.3, s=10, c='orange')
            axes[2, 1].set_xlabel('Gene Importance')
            axes[2, 1].set_ylabel('m/z Importance')
            imp_corr, _ = pearsonr(imp1[mask_imp], imp2[mask_imp])
            axes[2, 1].set_title(f'Importance Correlation (r={imp_corr:.3f})', fontweight='bold')
        
        # Difference map
        v1_norm = (v1 - np.nanmin(v1)) / (np.nanmax(v1) - np.nanmin(v1) + 1e-8)
        v2_norm = (v2 - np.nanmin(v2)) / (np.nanmax(v2) - np.nanmin(v2) + 1e-8)
        diff = v1_norm - v2_norm
        
        im_diff = axes[2, 2].imshow(diff, cmap='RdBu_r', origin='lower',
                                     extent=[x_min, x_max, y_min, y_max])
        axes[2, 2].set_title('Difference (Gene - m/z)', fontweight='bold')
        plt.colorbar(im_diff, ax=axes[2, 2])
        
        # Metrics
        metrics_text = f"""
Similarity Metrics (with 180° rotation):
─────────────────────────────────────
Combined Score:     {sim.get('combined_score', 0):.4f}
─────────────────────────────────────
Embedding Cosine:   {sim['embedding_cosine']:.4f}
CNN Cosine:         {sim['cnn_cosine']:.4f}
Transformer Cosine: {sim['transformer_cosine']:.4f}
GAT Cosine:         {sim['gat_cosine']:.4f}
Importance Overlap: {sim['importance_overlap']:.4f}
Raw Pearson:        {sim['raw_pearson']:.4f}
─────────────────────────────────────
Note: RNA rotated 180° to align with MSI
        """
        axes[2, 3].text(0.05, 0.95, metrics_text, transform=axes[2, 3].transAxes,
                       fontsize=10, verticalalignment='top', family='monospace',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        axes[2, 3].axis('off')
        
        plt.suptitle(f'Match: {gene_sig.feature_name} ↔ m/z {mz_sig.feature_name} (Aligned)',
                    fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        plt.close()
    
    def save_embeddings(self, gene_sig: SpatialSignature):
        """Save gene embeddings"""
        emb_data = {
            'global_embedding': gene_sig.global_embedding,
            'node_importance': gene_sig.node_importance,
            'coordinates': gene_sig.coordinates,
            'raw_values': gene_sig.raw_values,
            'cnn_features': gene_sig.cnn_features,
            'transformer_features': gene_sig.transformer_features,
            'gat_features': gene_sig.gat_features
        }
        
        save_path = os.path.join(
            self.output_dir, 'embeddings',
            f'{gene_sig.feature_name}_{gene_sig.sample_id}.pkl'
        )
        with open(save_path, 'wb') as f:
            pickle.dump(emb_data, f)
    
    def run_analysis(self, epochs: int = 150, top_k: int = 20):
        """Run complete analysis on ALL samples"""
        print("\n" + "="*70)
        print("GENE-TO-M/Z MATCHING V5 - HYBRID ARCHITECTURE")
        print("CNN + Transformer + GAT")
        print("="*70)
        
        gene_availability = self.check_gene_availability()
        
        # ===== TRAINING =====
        print("\n" + "-"*70)
        print("PHASE 1: Training Hybrid Models")
        print("-"*70)
        
        # RNA (6 neighbors - hexagonal)
        print("\nPreparing RNA data (6 neighbors)...")
        rna_train = {}
        rna_bio_imp = {}
        
        for sample_id in list(self.rna_data.keys())[:4]:
            adata = self.rna_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X[:, :50].toarray()
            else:
                features = adata.X[:, :50].copy()
            
            data, bio_imp = self.prepare_data(coords, features, n_neighbors=6)
            rna_train[sample_id] = data
            rna_bio_imp[sample_id] = bio_imp
            print(f"  {sample_id}: {data.x.shape}")
        
        self.rna_model = self.train_model(rna_train, rna_bio_imp, 'rna', epochs)
        
        # MSI (8 neighbors - Cartesian)
        print("\nPreparing MSI data (8 neighbors)...")
        msi_train = {}
        msi_bio_imp = {}
        
        for sample_id in list(self.msi_data.keys())[:4]:
            adata = self.msi_data[sample_id]
            coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
            
            if hasattr(adata.X, 'toarray'):
                features = adata.X[:, :50].toarray()
            else:
                features = adata.X[:, :50].copy()
            
            data, bio_imp = self.prepare_data(coords, features, n_neighbors=8)
            msi_train[sample_id] = data
            msi_bio_imp[sample_id] = bio_imp
            print(f"  {sample_id}: {data.x.shape}")
        
        self.msi_model = self.train_model(msi_train, msi_bio_imp, 'msi', epochs)
        
        # ===== MATCHING - ALL SAMPLES =====
        print("\n" + "-"*70)
        print("PHASE 2: Processing ALL Samples")
        print("-"*70)
        
        all_results = []
        
        for gene in AAD_TARGET_GENES:
            print(f"\n{'='*50}")
            print(f"Gene: {gene}")
            print(f"{'='*50}")
            
            # Get ALL available samples for this gene
            available = [s for s, a in gene_availability[gene].items() if a]
            
            if not available:
                print("  Not available")
                continue
            
            print(f"  Processing {len(available)} samples: {available}")
            
            for rna_sample in available:
                print(f"\n  Sample: {rna_sample}")
                
                # Extract gene signature
                gene_sig = self.extract_gene_signature(rna_sample, gene, self.rna_model)
                if gene_sig is None:
                    print("    Failed to extract")
                    continue
                
                # Save visualizations
                self.visualize_signature(
                    gene_sig,
                    os.path.join(self.output_dir, 'saliency_maps', f'{gene}_{rna_sample}.png')
                )
                
                # Save to gene_visualizations too
                self.visualize_signature(
                    gene_sig,
                    os.path.join(self.output_dir, 'gene_visualizations', f'{gene}_{rna_sample}.png')
                )
                
                # Save embeddings
                self.save_embeddings(gene_sig)
                
                # Find matching MSI
                msi_sample = rna_sample
                if msi_sample not in self.msi_data:
                    print(f"    MSI {msi_sample} not found")
                    continue
                
                print(f"    Matching vs MSI {msi_sample}...")
                
                adata = self.msi_data[msi_sample]
                coords = np.column_stack([adata.obs['x_um'].values, adata.obs['y_um'].values])
                
                if hasattr(adata.X, 'toarray'):
                    msi_data = adata.X.toarray()
                else:
                    msi_data = adata.X.copy()
                
                mz_names = list(adata.var_names)
                
                print(f"    Processing {len(mz_names)} m/z features...")
                mz_sigs = {}
                
                for i, mz_name in enumerate(mz_names):
                    mz_values = msi_data[:, i]
                    mz_sigs[mz_name] = self.extract_mz_signature(
                        msi_sample, mz_name, mz_values, coords, self.msi_model
                    )
                    
                    if (i + 1) % 100 == 0:
                        print(f"      {i+1}/{len(mz_names)}")
                
                # Find matches
                matches = self.find_matches(gene_sig, mz_sigs, top_k)
                all_results.append(matches)
                
                if len(matches) > 0:
                    print(f"\n    Top 5 matches:")
                    for idx in range(min(5, len(matches))):
                        m = matches.iloc[idx]
                        print(f"      {idx+1}. m/z {m['mz_feature']}: "
                              f"combined={m['combined_score']:.3f}")
                    
                    # Visualize top match
                    top_match = matches.iloc[0]
                    top_mz_sig = mz_sigs[top_match['mz_feature']]
                    
                    self.visualize_match(
                        gene_sig, top_mz_sig,
                        {k: top_match[k] for k in ['embedding_cosine', 'cnn_cosine',
                                                    'transformer_cosine', 'gat_cosine',
                                                    'importance_overlap', 'raw_pearson',
                                                    'combined_score']},
                        os.path.join(self.output_dir, 'match_visualizations',
                                    f'{gene}_{rna_sample}_top_match.png')
                    )
        
        # Save results
        if all_results:
            results = pd.concat(all_results, ignore_index=True)
            results = results.sort_values('combined_score', ascending=False)
            
            output_file = os.path.join(self.output_dir, 'gene_to_mz_matches_v5.csv')
            results.to_csv(output_file, index=False)
            print(f"\n\nResults saved to: {output_file}")
            
            # Summary
            print("\n" + "="*70)
            print("SUMMARY - TOP MATCH PER GENE")
            print("="*70)
            
            for gene in AAD_TARGET_GENES:
                gene_results = results[results['gene'] == gene]
                if len(gene_results) > 0:
                    top = gene_results.iloc[0]
                    print(f"\n{gene}:")
                    print(f"  Best m/z: {top['mz_feature']}")
                    print(f"  Sample: {top['rna_sample']}")
                    print(f"  Combined: {top['combined_score']:.3f}")
            
            return results
        
        return None

In [11]:

def main():
    print("="*70)
    print("GENE-TO-M/Z MATCHING V5")
    print("Hybrid: CNN + Transformer + GAT")
    print("="*70)
    
    matcher = HybridMatcher(output_dir='./gene_to_mz_results_v5')
    matcher.load_all_data()
    
    results = matcher.run_analysis(epochs=150, top_k=20)
    
    print("\n" + "="*70)
    print("COMPLETE!")
    print("="*70)
    print(f"\nOutput: {matcher.output_dir}/")
    print("  - saliency_maps/: All gene spatial patterns")
    print("  - gene_visualizations/: Gene visualizations")
    print("  - match_visualizations/: Top matches")
    print("  - embeddings/: Saved embeddings (pkl)")
    print("  - training_curves/: Loss plots")
    
    return matcher, results

In [12]:
if __name__ == "__main__":
    matcher, results = main()

GENE-TO-M/Z MATCHING V5
Hybrid: CNN + Transformer + GAT
Loading RNA-seq data...
  YC_1: (2112, 800)
  YC_2: (2775, 800)
  YC_3: (2808, 800)
  YC_4: (2725, 800)
  YAD_1: (2915, 800)
  YAD_2: (2960, 800)
  YAD_3: (2880, 800)
  YAD_4: (2939, 800)
  AC_1: (3065, 800)
  AC_2: (3054, 800)
  AC_3: (2892, 800)
  AC_4: (3002, 800)
  AAD_1: (2700, 800)
  AAD_2: (2171, 800)
  AAD_3: (1584, 800)
  AAD_4: (2438, 800)

Loading MSI data...
  YC_1: (6688, 528)
  YC_2: (7858, 528)
  YC_3: (7150, 528)
  YC_4: (6067, 528)
  YAD_1: (7517, 528)
  YAD_2: (7596, 528)
  YAD_3: (6844, 528)
  YAD_4: (7591, 528)
  AC_1: (6955, 528)
  AC_2: (5729, 528)
  AC_3: (7569, 528)
  AC_4: (7792, 528)
  AAD_1: (6471, 528)
  AAD_2: (5959, 528)
  AAD_3: (5392, 528)
  AAD_4: (6833, 528)

GENE-TO-M/Z MATCHING V5 - HYBRID ARCHITECTURE
CNN + Transformer + GAT

Gene availability:
  Thy1: 16/16 samples
  App: 16/16 samples
  Apoe: 16/16 samples
  Mapt: 16/16 samples

----------------------------------------------------------------

KeyboardInterrupt: 